In [2]:
# Librerías necesarias para el análisis exploratorio
import pandas as pd

In [4]:
# Cargamos el dataset original
df = pd.read_csv('../hr.csv')

# Confirmamos que se ha cargado correctamente
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()

Filas: 1474
Columnas: 35


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,NaN,0,7,3.0,3,0,0,0,0.0
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80.0,0,8,3.0,3,8,7,3,0.0
4,27.0,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80.0,1,6,3.0,3,2,2,2,2.0


### Comprobación de Integridad del Archivo

In [10]:
# 1. Validamos las dimensiones exactas del dataset
print(f"¿El número de filas es el esperado? {'Sí' if df.shape[0] == 1474 else 'No'} ({df.shape[0]} filas)")
print(f"¿El número de columnas es el esperado? {'Sí' if df.shape[1] == 35 else 'No'} ({df.shape[1]} columnas)")

# 2. Verificamos que no haya problemas de lectura
print(f"\nTotal de celdas con datos: {df.notnull().sum().sum()}")
print(f"Total de celdas vacías (nulos): {df.isnull().sum().sum()}")

# 3. Inspección visual rápida para validar que el separador es correcto
print("\n--- Vista de control de las 3 primeras filas ---")
display(df.head(3))

¿El número de filas es el esperado? Sí (1474 filas)
¿El número de columnas es el esperado? Sí (35 columnas)

Total de celdas con datos: 50694
Total de celdas vacías (nulos): 896

--- Vista de control de las 3 primeras filas ---


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,NaN,0,7,3.0,3,0,0,0,0.0


### número exacto de empleados en cada situación

In [5]:
# Contamos cuántos empleados se fueron (Yes) y cuántos se quedaron (No)
df['Attrition'].value_counts()

Attrition
No     1236
Yes     238
Name: count, dtype: int64

### lo pasamos a formato porcentaje

In [6]:
# Calculamos el porcentaje de cada valor
df['Attrition'].value_counts(normalize=True).mul(100).round(2)

Attrition
No     83.85
Yes    16.15
Name: proportion, dtype: float64

**Conclusión:** El 16.15% de los empleados han abandonado la empresa. 
Aunque no es una crisis absoluta, confirma una fuga de talento real. 
Además hay un desbalanceo de clases (83.85% No vs 16.15% Yes), 
algo importante a tener en cuenta para un modelo predictivo.

### Revisión de columnas clave

In [7]:
# Revisamos los valores únicos de las columnas categóricas clave
# para detectar errores de formato, erratas o valores inesperados
columnas_categoricas = ['JobRole', 'Department', 'Gender', 'MaritalStatus', 'OverTime']

for col in columnas_categoricas:
    print(f"--- Valores únicos en la columna: {col} ---")
    print(df[col].unique())
    print()


--- Valores únicos en la columna: JobRole ---
<StringArray>
[          ' sALES eXECUTIVE ',        ' rESEARCH sCIENTIST ',
     ' lABORATORY tECHNICIAN ',    ' mANUFACTURING dIRECTOR ',
 ' hEALTHCARE rEPRESENTATIVE ',                   ' mANAGER ',
      ' sALES rEPRESENTATIVE ',         ' rESEARCH dIRECTOR ',
           ' hUMAN rESOURCES ']
Length: 9, dtype: str

--- Valores únicos en la columna: Department ---
<StringArray>
['Sales', 'Research & Development', nan, 'Human Resources']
Length: 4, dtype: str

--- Valores únicos en la columna: Gender ---
<StringArray>
['Female', 'Male']
Length: 2, dtype: str

--- Valores únicos en la columna: MaritalStatus ---
<StringArray>
['Single', 'Married', 'Divorced', nan, 'Marreid']
Length: 5, dtype: str

--- Valores únicos en la columna: OverTime ---
<StringArray>
['Yes', 'No', nan]
Length: 3, dtype: str



**Hallazgos detectados:**

- **`JobRole`:** Formato de texto incorrecto (ej. `' sALES eXECUTIVE '`) y espacios en blanco al inicio y al final.
- **`Department`:** Contiene valores nulos. Hay empleados cuyo departamento se desconoce.
- **`MaritalStatus`:** Contiene valores nulos y una errata (`'Marreid'` en vez de `'Married'`), lo que genera dos grupos para un mismo valor.
- **`OverTime`:** Contiene valores nulos.
- **`Gender`:** Sin problemas detectados.

### Columnas constantes

In [8]:
# Contamos los valores únicos de cada columna ordenados de menor a mayor
# Las columnas con valor 1 son constantes y no aportan información al análisis
df.nunique().sort_values()

EmployeeCount                  1
Over18                         1
StandardHours                  1
Attrition                      2
OverTime                       2
PerformanceRating              2
Gender                         2
BusinessTravel                 3
Department                     3
JobSatisfaction                4
RelationshipSatisfaction       4
StockOptionLevel               4
MaritalStatus                  4
EnvironmentSatisfaction        4
JobInvolvement                 4
WorkLifeBalance                4
Education                      5
JobLevel                       5
EducationField                 6
TrainingTimesLastYear          7
JobRole                        9
NumCompaniesWorked            10
PercentSalaryHike             15
YearsSinceLastPromotion       16
YearsWithCurrManager          18
YearsInCurrentRole            19
DistanceFromHome              29
YearsAtCompany                37
TotalWorkingYears             40
Age                           43
HourlyRate

In [9]:
# Comprobamos si hay filas duplicadas en el dataset
print(f"Filas duplicadas: {df.duplicated().sum()}")

Filas duplicadas: 4


**Hallazgos detectados:**

- **`EmployeeCount`**, **`Over18`** y **`StandardHours`** tienen un único valor único. No aportan información y se eliminarán en la Fase 2.
- Se han detectado **4 filas duplicadas** en el dataset. Se eliminarán en la Fase 2.

## 📋 Reporte de hallazgos — Fase 1 (Pareja B)

**1. Fuga de talento (variable objetivo):**
- El **16.15%** de los empleados han abandonado la empresa (`Attrition = Yes`).
- Hay un desbalanceo de clases (83.85% No vs 16.15% Yes) importante para un modelo predictivo.

**2. Problemas en columnas categóricas:**
- **`JobRole`:** Formato incorrecto (ej. `' sALES eXECUTIVE '`) y espacios en blanco al inicio y final.
- **`Department`:** Contiene valores nulos.
- **`MaritalStatus`:** Contiene valores nulos y una errata (`'Marreid'` en vez de `'Married'`).
- **`OverTime`:** Contiene valores nulos.

**3. Problemas en columnas numéricas:**
- **Nulos detectados en:** `Age` (73), `JobSatisfaction` (29), `MonthlyIncome` (14), `TrainingTimesLastYear` (88), `YearsWithCurrManager` (148).
- **Tipos incorrectos:** `Age`, `JobSatisfaction`, `MonthlyIncome`, `TrainingTimesLastYear` y `YearsWithCurrMan